TASK 1 

Class for temperatures and conversions between C, K and F. 

In [2]:


class Temperature:

    # Tolerance for floating-point comparisons
    TOLERANCE = 1e-9



    def __init__(self, value, unit ="C"):
        """Initialize a Temperature instance with a value and unit."""
        unit = unit.upper()

        # Specify the valid units for this class
        valid_units = ['C', 'F', 'K']

        # Check if the unit is valid, if not raise a ValueError
        if unit not in valid_units:
            raise ValueError("Invalid temperature unit. Please use 'C', 'F', or 'K'.")

        # Absolute zero check for each unit, if the value is below absolute zero raise a ValueError
        if (
            (unit == 'C' and value < -273.15)
            or (unit == 'F' and value < -459.67)
            or (unit == 'K' and value < 0)
        ): raise ValueError("Temperature cannot be below absolute zero.")
        
        self._value = float(value)
        self._unit = unit



    # Define read-only properties for value and unit
    @property
    def value(self):
        return self._value

    @property
    def unit(self):
        return self._unit
    

    
    # Converts values from F and K to C 
    def to_celsius(self) -> float:
        if self.unit == 'F':
            return (self.value - 32) * (5/9)
        
        elif self.unit == 'K':
            return (self.value - 273.15)
        
        else:
            return self.value

    # Converts values from C and K to F 
    def to_fahrenheit(self) -> float:
        if self.unit == 'C':
            return (self.value * (9/5)) + 32
        
        elif self.unit == 'K':
            return (self.value - 273.15) * (9/5) + 32
        
        else:
            return self.value

    # Converts values from F and C to K 
    def to_kelvin(self) -> float:
        if self.unit == 'F':
            result = (self.value - 32) * (5/9) + 273.15
            # Check if the result is below absolute zero, if so return 0.0
            if self.TOLERANCE <= result < 0:
                return 0.0
            return result
        
        elif self.unit == 'C':
            return self.value + 273.15
        
        else:
            return self.value

  

    # Function identifies what the target unit is and 
    # calls the appropriate method to calculate new temp
    def convert_to(self, target_unit: str) -> 'Temperature':
        if target_unit.upper() == 'C':
            new_temp = self.to_celsius()
        elif target_unit.upper() == 'F':
            new_temp = self.to_fahrenheit()
        elif target_unit.upper() == 'K':
            new_temp = self.to_kelvin()
        else:
            raise ValueError("Invalid temperature unit. Please use 'C', 'F', or 'K'.")

        return Temperature(new_temp, target_unit.upper())



    # Define string representation methods for the class
    def __repr__(self) -> str:
        return f'Temperature({self.value}, {self.unit!r})'

    def __str__(self) -> str:
        if self.unit == 'K':
            return f'{self.value:.2f} K'
        else:
            return f'{self.value:.2f} °{self.unit}'



    # Define six comparison methods for the class
    # Comparisons ==
    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return abs(self.to_kelvin() - other.to_kelvin()) <= self.TOLERANCE

    # Comparisons !=
    def __ne__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return not self.__eq__(other)

    # Comparisons <
    def __lt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return self.to_kelvin() < other.to_kelvin() and self.__ne__(other)

    # Comparisons <=
    def __le__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return self.__lt__(other) or self.__eq__(other)

    # Comparisons >
    def __gt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return self.to_kelvin() > other.to_kelvin() and self.__ne__(other)

    # Comparisons >=
    def __ge__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented

        return self.__gt__(other) or self.__eq__(other)


Task 1b: Test temperature script 

In [6]:
import pytest
from temperature_class import Temperature

TOLERANCE = Temperature.TOLERANCE


# 1. Initialization tests
def test_default_unit():
    t = Temperature(25)
    assert t.unit == "C"
    assert t.value == 25


def test_case_insensitive():
    assert Temperature(25, "c").unit == "C"
    assert Temperature(25, "f").unit == "F"
    assert Temperature(25, "k").unit == "K"


def test_read_only():
    t = Temperature(25, "C")

    with pytest.raises(AttributeError):
        t.value = 30

    with pytest.raises(AttributeError):
        t.unit = "F"


# 2. Validation tests
def test_invalid_unit():
    with pytest.raises(ValueError):
        Temperature(25, "X")


def test_below_absolute_zero():
    with pytest.raises(ValueError):
        Temperature(-274, "C")

    with pytest.raises(ValueError):
        Temperature(-460, "F")

    with pytest.raises(ValueError):
        Temperature(-1, "K")


# 3. Conversion tests
@pytest.mark.parametrize(
    "C, F, K",
    [
        (-273.15, -459.67, 0),    # Absolute zero
        (-40, -40, 233.15),       # Celsius-Fahrenheit crossover
        (0, 32, 273.15),          # Freezing point of water
        (37, 98.6, 310.15),       # Human body temperature
        (100, 212, 373.15),       # Boiling point of water
    ],
)
def test_conversion(C, F, K):
    t_c = Temperature(C, "C")
    assert abs(t_c.to_fahrenheit() - F) <= TOLERANCE
    assert abs(t_c.to_kelvin() - K) <= TOLERANCE

    t_f = Temperature(F, "F")
    assert abs(t_f.to_celsius() - C) <= TOLERANCE
    assert abs(t_f.to_kelvin() - K) <= TOLERANCE

    t_k = Temperature(K, "K")
    assert abs(t_k.to_celsius() - C) <= TOLERANCE
    assert abs(t_k.to_fahrenheit() - F) <= TOLERANCE


# 4. Round-trip conversion tests ("C" -> "F" -> "K" -> "C")
@pytest.mark.parametrize("C", [-273.15, -40, 0, 37, 100])
def test_round_trip_conversion(C):
    t_c = Temperature(C, "C")
    t_f = Temperature(t_c.to_fahrenheit(), "F")
    t_k = Temperature(t_f.to_kelvin(), "K")
    round_tripped_c = t_k.to_celsius()

    assert abs(round_tripped_c - C) <= TOLERANCE


# 5. Cross-unit comparisons
def test_comparisons_across_units():
    freezing_c = Temperature(0, "C")
    freezing_f = Temperature(32, "F")
    boiling_k = Temperature(373.15, "K")
    warm_f = Temperature(68, "F")      # 20°C

    # Equality
    assert freezing_c == freezing_f
    assert not (freezing_c == boiling_k)

    # Strict inequalities
    assert freezing_c < boiling_k
    assert freezing_f < warm_f
    assert boiling_k > freezing_c
    assert warm_f > freezing_f

    # Non-strict inequalities
    assert freezing_c <= freezing_f
    assert freezing_c <= boiling_k
    assert boiling_k >= freezing_c
    assert freezing_f >= freezing_c


# 6. Formatting tests
def test_str_and_repr():
    t_c = Temperature(25, "C")
    t_f = Temperature(77, "F")
    t_k = Temperature(300, "K")

    assert str(t_c) == "25.00 °C"
    assert str(t_f) == "77.00 °F"
    assert str(t_k) == "300.00 K"
    assert repr(t_c) == "Temperature(25.0, 'C')"

Testing output:

test_temperature.py::test_default_unit PASSED                                                      [  5%]
test_temperature.py::test_case_insensitive PASSED                                                  [ 11%]
test_temperature.py::test_read_only PASSED                                                         [ 17%]
test_temperature.py::test_invalid_unit PASSED                                                      [ 23%]
test_temperature.py::test_below_absolute_zero PASSED                                               [ 29%]
test_temperature.py::test_conversion[-273.15--459.67-0] PASSED                                     [ 35%]
test_temperature.py::test_conversion[-40--40-233.15] PASSED                                        [ 41%]
test_temperature.py::test_conversion[0-32-273.15] PASSED                                           [ 47%]
test_temperature.py::test_conversion[37-98.6-310.15] PASSED                                        [ 52%]
test_temperature.py::test_conversion[100-212-373.15] PASSED                                        [ 58%]
test_temperature.py::test_round_trip_conversion[-273.15] PASSED                                    [ 64%]
test_temperature.py::test_round_trip_conversion[-40] PASSED                                        [ 70%]
test_temperature.py::test_round_trip_conversion[0] PASSED                                          [ 76%]
test_temperature.py::test_round_trip_conversion[37] PASSED                                         [ 82%]
test_temperature.py::test_round_trip_conversion[100] PASSED                                        [ 88%]
test_temperature.py::test_comparisons_across_units PASSED                                          [ 94%]
test_temperature.py::test_str_and_repr PASSED                                                      [100%]

========================================== 17 passed in 0.29s ===========================================



Task 2a : Fraction class 

In [7]:
import math


class Fraction:
    """Represents a rational number with integer numerator and strictly positive denominator."""

    def __init__(self, numerator: int, denominator: int = 1):
        if isinstance(numerator, bool) or not isinstance(numerator, int):
            raise TypeError("Numerator must be an integer.")
        if isinstance(denominator, bool) or not isinstance(denominator, int):
            raise TypeError("Denominator must be an integer.")
        if denominator <= 0:
            raise ValueError("Denominator must be strictly positive (> 0).")

        common = math.gcd(numerator, denominator)
        self._numerator = numerator // common
        self._denominator = denominator // common

    @property
    def numerator(self) -> int:
        return self._numerator

    @property
    def denominator(self) -> int:
        return self._denominator

    #String representations 
    def __str__(self) -> str:
        if self._denominator == 1:
            return str(self._numerator)
        return f"{self._numerator}/{self._denominator}"

    def __repr__(self) -> str:
        return f"Fraction({self._numerator}, {self._denominator})"

    # Auxiliary function to determine if input 'other' is a fraction, and coerce it into being
    # a fraction if it is an integer. 
    @staticmethod
    def _coerce(other):
        if isinstance(other, Fraction):
            return other
        if isinstance(other, int) and not isinstance(other, bool):
            return Fraction(other, 1)
        return NotImplemented

    # All 6 comparison methods
    def __eq__(self, other) -> bool:
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return False
        return (
            self._numerator == other_frac._numerator
            and self._denominator == other_frac._denominator
        )

    def __ne__(self, other) -> bool:
        return not (self == other)

    def __lt__(self, other) -> bool:
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        return self._numerator * other_frac._denominator < other_frac._numerator * self._denominator

    def __le__(self, other) -> bool:
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        return self._numerator * other_frac._denominator <= other_frac._numerator * self._denominator

    def __gt__(self, other) -> bool:
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        return self._numerator * other_frac._denominator > other_frac._numerator * self._denominator

    def __ge__(self, other) -> bool:
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        return self._numerator * other_frac._denominator >= other_frac._numerator * self._denominator

    # Additional 'special' methods for fraction addition, subtraction, multiplication and division
    def __add__(self, other) -> "Fraction":
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        num = self._numerator * other_frac._denominator + other_frac._numerator * self._denominator
        den = self._denominator * other_frac._denominator
        return Fraction(num, den)

    def __sub__(self, other) -> "Fraction":
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        num = self._numerator * other_frac._denominator - other_frac._numerator * self._denominator
        den = self._denominator * other_frac._denominator
        return Fraction(num, den)

    def __mul__(self, other) -> "Fraction":
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        return Fraction(
            self._numerator * other_frac._numerator,
            self._denominator * other_frac._denominator,
        )

    def __truediv__(self, other) -> "Fraction":
        other_frac = self._coerce(other)
        if other_frac is NotImplemented:
            return NotImplemented
        if other_frac._numerator == 0:
            raise ZeroDivisionError("Cannot divide by a zero fraction.")
        num = self._numerator * other_frac._denominator
        den = self._denominator * other_frac._numerator
        if den < 0:
            num, den = -num, -den
        return Fraction(num, den)
    
    def __neg__(self) -> "Fraction":
        return Fraction(-self._numerator, self._denominator)

    def __abs__(self) -> "Fraction":
        return Fraction(abs(self._numerator), self._denominator)

    def __float__(self) -> float:
        return self._numerator / self._denominator



Task 2b: Test fractions script 

In [ ]:
import pytest
from fraction_class import Fraction


def test_init_and_reduction():
    """Test initialization, default denominator, and automatic fraction reduction."""
    assert Fraction(2, 4).numerator == 1
    assert Fraction(2, 4).denominator == 2
    assert Fraction(-6, 9).numerator == -2
    assert Fraction(-6, 9).denominator == 3
    assert Fraction(5).numerator == 5
    assert Fraction(5).denominator == 1
    assert Fraction(0, 7).numerator == 0
    assert Fraction(0, 7).denominator == 1


# Reject non-integer inputs (including booleans) with TypeError
@pytest.mark.parametrize("bad_value", [1.5, "2", None, True, False])
def test_non_integer_arguments(bad_value):
    with pytest.raises(TypeError):
        Fraction(bad_value, 2)
    with pytest.raises(TypeError):
        Fraction(2, bad_value)


# Reject zero or negative denominators with ValueError
@pytest.mark.parametrize("bad_den", [0, -1, -5])
def test_non_positive_denominator(bad_den):
    with pytest.raises(ValueError):
        Fraction(1, bad_den)


def test_comparisons():
    """Test all six comparison operators between fractions and integers."""
    a = Fraction(1, 2)
    b = Fraction(2, 4)
    c = Fraction(1, 3)
    neg = Fraction(-1, 2)

    # Equality and inequality
    assert a == b
    assert a != c
    assert Fraction(4, 2) == 2

    # Ordering (<, <=, >, >=)
    assert neg < c < a
    assert c <= a
    assert a <= b
    assert a > c > neg
    assert a >= c
    assert a >= b


def test_string_representations():
    """Test str() and repr() outputs."""
    assert str(Fraction(3, 4)) == "3/4"
    assert str(Fraction(6, 3)) == "2"
    assert repr(Fraction(2, 4)) == "Fraction(1, 2)"


def test_arithmetic_and_unary():
    """Test arithmetic operators, unary operations, float conversion, and zero division."""
    x = Fraction(1, 2)
    y = Fraction(1, 3)

    # Basic arithmetic (+, -, *, /)
    assert x + y == Fraction(5, 6)
    assert x - y == Fraction(1, 6)
    assert x * y == Fraction(1, 6)
    assert x / y == Fraction(3, 2)

    # Unary operators and float conversion
    assert -x == Fraction(-1, 2)
    assert abs(Fraction(-3, 4)) == Fraction(3, 4)
    assert float(Fraction(1, 4)) == pytest.approx(0.25)

    # Division by zero
    with pytest.raises(ZeroDivisionError):
        _ = x / Fraction(0, 1)

Testing output:

test_fractions.py::test_init_and_reduction PASSED                                                  [  8%]
test_fractions.py::test_non_integer_arguments[1.5] PASSED                                          [ 16%]
test_fractions.py::test_non_integer_arguments[2] PASSED                                            [ 25%]
test_fractions.py::test_non_integer_arguments[None] PASSED                                         [ 33%]
test_fractions.py::test_non_integer_arguments[True] PASSED                                         [ 41%]
test_fractions.py::test_non_integer_arguments[False] PASSED                                        [ 50%]
test_fractions.py::test_non_positive_denominator[0] PASSED                                         [ 58%]
test_fractions.py::test_non_positive_denominator[-1] PASSED                                        [ 66%]
test_fractions.py::test_non_positive_denominator[-5] PASSED                                        [ 75%]
test_fractions.py::test_comparisons PASSED                                                         [ 83%]
test_fractions.py::test_string_representations PASSED                                              [ 91%]
test_fractions.py::test_arithmetic_and_unary PASSED                                                [100%]

========================================== 12 passed in 0.08s ===========================================

